# 00 — Dataset Preparation
**Knowledge Distillation + Quantization for Compact Models**

This notebook downloads GLUE SST-2, inspects it, tokenises it, exports
portable Parquet files, validates them, and packages them for download.

---

## ⚠ Run this notebook on Google Colab

The local Windows machine has a WDAC policy that blocks `pyarrow` native
DLLs, which `datasets` and `evaluate` depend on. All export steps must run
on Colab. After completing this notebook, download `processed_data.zip` and
extract it into your local `data/processed/` folder.

---

## How to open this notebook in Colab

**Step 1 — Upload the project ZIP to Colab**

On your local machine, create `colab_upload.zip` by running:
```
python scripts/make_colab_zip.py
```
This packages `src/`, `notebooks/`, and `requirements.txt` into a single zip.

**Step 2 — Open Colab**

1. Go to https://colab.research.google.com
2. File → Upload notebook → select this file (`00_dataset_preparation.ipynb`)
3. Runtime → Change runtime type → T4 GPU → Save

**Step 3 — Upload the project ZIP**

In the first cell below, choose **Upload ZIP** mode and follow the prompt to
upload `colab_upload.zip` when asked.

**Step 4 — Run all cells** (Runtime → Run all)

**Step 5 — Download the results**

The final cell creates `processed_data.zip` and triggers a browser download.
Extract its contents into your local `data/processed/` directory.


---
## Step 1 — Runtime detection

In [ ]:
import sys, os, importlib

# Colab detection: check for the colab module without triggering an import error
IN_COLAB = importlib.util.find_spec('google.colab') is not None

print(f'Running in Colab : {IN_COLAB}')
print(f'Python version   : {sys.version}')

if IN_COLAB:
    import subprocess, sys as _sys
    r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                        '--format=csv,noheader'],
                       capture_output=True, text=True)
    if r.returncode == 0:
        print(f'GPU              : {r.stdout.strip()}')
    else:
        print('GPU              : not detected (CPU runtime)')

---
## Step 2 — Install dependencies

In [ ]:
# Install packages needed for dataset preparation only.
# torch is pre-installed on Colab; we do NOT reinstall it to
# avoid CUDA version conflicts.
import subprocess, sys

PACKAGES = [
    'transformers>=4.38.0',
    'datasets>=2.18.0',
    'accelerate>=0.27.0',
    'tokenizers>=0.15.0',
    'huggingface_hub>=0.21.0',
    'pyarrow>=14.0.0',
    'pandas>=2.0.0',
    'numpy>=1.24.0',
    'scikit-learn>=1.3.0',
    'tqdm>=4.66.0',
]

if IN_COLAB:
    print('Installing packages…')
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet'] + PACKAGES,
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print('pip STDERR:', result.stderr[-2000:])
        raise RuntimeError('Package installation failed.')
    print('Installation complete.')
else:
    print('Local run — using existing .venv311 environment.')

# Verify critical imports
import_checks = {}
for pkg in ['datasets', 'transformers', 'pyarrow', 'pandas', 'numpy']:
    try:
        mod = __import__(pkg)
        import_checks[pkg] = getattr(mod, '__version__', 'ok')
    except ImportError as e:
        import_checks[pkg] = f'FAILED: {e}'

print('\nPackage versions after install:')
for pkg, ver in import_checks.items():
    status = 'OK' if 'FAILED' not in str(ver) else 'FAIL'
    print(f'  [{status}]  {pkg:<15} {ver}')

if any('FAILED' in str(v) for v in import_checks.values()):
    raise RuntimeError('One or more critical packages failed to import. '
                       'Check the output above.')

---
## Step 3 — Upload project source and set PROJECT_ROOT

Choose one of the three options below by setting `UPLOAD_MODE`:

| Mode | When to use |
|------|-------------|
| `'zip'` | You have `colab_upload.zip` on your local machine (recommended) |
| `'drive'` | The project folder is in your Google Drive |
| `'inline'` | You want the notebook to write `src/` files itself (no upload needed) |

For the **zip** mode, run `python scripts/make_colab_zip.py` locally first to
create `colab_upload.zip`, then upload it when prompted.

In [ ]:
# ── CONFIGURATION — change this if needed ────────────────────────────────
UPLOAD_MODE = 'zip'   # 'zip' | 'drive' | 'inline'

# Only needed for 'drive' mode:
DRIVE_PROJECT_PATH = 'MyDrive/knowledge-distillation-quantization'
# ─────────────────────────────────────────────────────────────────────────

from pathlib import Path

PROJECT_ROOT = None

if not IN_COLAB:
    # Local: notebook is in notebooks/, so project root is one level up
    PROJECT_ROOT = str(Path('..').resolve())
    print(f'Local run. PROJECT_ROOT = {PROJECT_ROOT}')

elif UPLOAD_MODE == 'zip':
    from google.colab import files as _colab_files
    import zipfile, shutil

    print('Please upload colab_upload.zip when the file picker appears.')
    print('(Generate it locally with: python scripts/make_colab_zip.py)')

    uploaded = _colab_files.upload()
    if not uploaded:
        raise RuntimeError('No file was uploaded. Please re-run this cell.')

    zip_name = list(uploaded.keys())[0]
    print(f'Uploaded: {zip_name}  ({len(uploaded[zip_name]):,} bytes)')

    extract_dir = '/content/kdq'
    if Path(extract_dir).exists():
        shutil.rmtree(extract_dir)
    Path(extract_dir).mkdir()

    with zipfile.ZipFile(zip_name, 'r') as zf:
        zf.extractall(extract_dir)

    PROJECT_ROOT = extract_dir
    print(f'Extracted to {PROJECT_ROOT}')
    print('Contents:', [p.name for p in Path(PROJECT_ROOT).iterdir()])

elif UPLOAD_MODE == 'drive':
    from google.colab import drive as _drive
    _drive.mount('/content/drive')
    PROJECT_ROOT = f'/content/drive/{DRIVE_PROJECT_PATH}'
    if not Path(PROJECT_ROOT).exists():
        raise FileNotFoundError(
            f'Drive path not found: {PROJECT_ROOT}. '
            'Check DRIVE_PROJECT_PATH above.'
        )
    print(f'Drive mounted. PROJECT_ROOT = {PROJECT_ROOT}')

elif UPLOAD_MODE == 'inline':
    # Write the minimum src/ files directly into /content/kdq
    PROJECT_ROOT = '/content/kdq'
    for d in ['src', 'data/processed', 'results']:
        Path(f'{PROJECT_ROOT}/{d}').mkdir(parents=True, exist_ok=True)
    print(f'Inline mode: created {PROJECT_ROOT}/')
    print('NOTE: src/ files must be uploaded manually via the Files panel.')
    print('      Or switch UPLOAD_MODE to \'zip\' for automatic setup.')

else:
    raise ValueError(f'Unknown UPLOAD_MODE: {UPLOAD_MODE!r}. '
                     'Set to \'zip\', \'drive\', or \'inline\'.')

assert PROJECT_ROOT is not None, 'PROJECT_ROOT was not set.'

# Ensure data/processed directory exists
PROCESSED_DIR = Path(PROJECT_ROOT) / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Add project root to sys.path so `from src import ...` works
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f'\nPROJECT_ROOT  = {PROJECT_ROOT}')
print(f'PROCESSED_DIR = {PROCESSED_DIR}')
print(f'sys.path[0]   = {sys.path[0]}')

---
## Step 4 — Verify source files and import project modules

In [ ]:
# Check that all required src/ files are present before importing
REQUIRED_SRC = ['__init__.py', 'config.py', 'dataset.py', 'utils.py']
src_dir = Path(PROJECT_ROOT) / 'src'
missing = [f for f in REQUIRED_SRC if not (src_dir / f).exists()]

if missing:
    raise FileNotFoundError(
        f'Missing src/ files: {missing}\n'
        'Make sure the ZIP was uploaded correctly or switch to \'drive\' mode.'
    )

print('src/ file check:')
for f in REQUIRED_SRC:
    print(f'  FOUND  src/{f}')

# Now safe to import
from src import config
import src.dataset as ds

print('\nProject modules imported successfully.')
config.print_config()

---
## Step 5 — Download and inspect GLUE SST-2

In [ ]:
# SUBSET_SIZE controls how many examples are used.
#   None  → full dataset  (67,349 train / 872 validation)  ← use for final export
#   200   → quick test run (useful for debugging)
SUBSET_SIZE = None

print(f'Loading GLUE SST-2 (subset_size={SUBSET_SIZE})…')
dataset = ds.load_sst2(subset_size=SUBSET_SIZE, seed=config.RANDOM_SEED)

print(f'\nLoaded splits  : {list(dataset.keys())}')
print(f'Train examples : {len(dataset["train"]):,}')
print(f'Val examples   : {len(dataset["validation"]):,}')
print(f'Columns        : {dataset["train"].column_names}')
print(f'\nFirst train example:')
print(f'  sentence : {dataset["train"][0]["sentence"]}')
print(f'  label    : {dataset["train"][0]["label"]}')
print(f'  idx      : {dataset["train"][0]["idx"]}')

In [ ]:
# Detailed inspection: class balance and sample sentences
summary = ds.inspect_dataset(dataset)

---
## Step 6 — Tokenisation demonstration

In [ ]:
# Load BERT-base-uncased tokenizer
tokenizer = ds.get_tokenizer(config.TEACHER_MODEL_NAME)
print(f'Tokenizer class : {tokenizer.__class__.__name__}')
print(f'Vocabulary size : {tokenizer.vocab_size:,}')
print(f'Model max length: {tokenizer.model_max_length}')

# Single-example demonstration
example = dataset['train'][0]
encoded = tokenizer(
    example['sentence'],
    truncation=True,
    max_length=config.MAX_SEQ_LENGTH
)
print(f'\nSentence  : {example["sentence"]}')
print(f'Label     : {example["label"]}  ({config.LABEL_NAMES[example["label"]]})')
print(f'Token count     : {len(encoded["input_ids"])}')
print(f'First 20 token ids : {encoded["input_ids"][:20]}')
print(f'First 20 tokens    : {tokenizer.convert_ids_to_tokens(encoded["input_ids"][:20])}')

In [ ]:
# Tokenise the entire dataset (no padding — dynamic padding at collation time)
tokenized_dataset = ds.tokenize_dataset(dataset, tokenizer, config.MAX_SEQ_LENGTH)

print('Tokenised dataset:')
print(tokenized_dataset)

# Spot-check: decode 3 random samples
ok = ds.validate_tokenization(tokenized_dataset, tokenizer, n_samples=3)
print(f'\nTokenisation spot-check passed: {ok}')

In [ ]:
# Sequence length statistics
import numpy as np

# input_ids in torch format are tensors; convert to int lists for len()
train_lengths = [len(x) for x in tokenized_dataset['train']['input_ids']]
val_lengths   = [len(x) for x in tokenized_dataset['validation']['input_ids']]

print('Train sequence lengths:')
print(f'  count  = {len(train_lengths):,}')
print(f'  min    = {min(train_lengths)}')
print(f'  max    = {max(train_lengths)}')
print(f'  mean   = {np.mean(train_lengths):.1f}')
print(f'  median = {np.median(train_lengths):.0f}')
print(f'  p95    = {np.percentile(train_lengths, 95):.0f}')
print(f'  p99    = {np.percentile(train_lengths, 99):.0f}')

print('\nValidation sequence lengths:')
print(f'  count  = {len(val_lengths):,}')
print(f'  min    = {min(val_lengths)}')
print(f'  max    = {max(val_lengths)}')
print(f'  mean   = {np.mean(val_lengths):.1f}')
print(f'  median = {np.median(val_lengths):.0f}')

---
## Step 7 — Export processed dataset to Parquet (CSV fallback)

In [ ]:
# Export raw sentences + labels + IDs
# We do NOT store tokenised sequences — they are recomputed at training time.
# This keeps files small and re-tokenisable with any tokenizer.

export_paths = ds.export_processed(
    dataset,
    output_dir=PROCESSED_DIR,
    split_meta={'inspection_summary': summary},
)

print('\nExported files:')
for key, path in export_paths.items():
    p = Path(str(path))
    if p.exists():
        size_kb = p.stat().st_size / 1024
        print(f'  {key:<20} {p.name:<30} ({size_kb:.1f} KB)')
    else:
        print(f'  {key:<20} NOT FOUND — export may have failed')

---
## Step 8 — Validation

In [ ]:
# Validate the raw HuggingFace dataset
print('=' * 60)
print('Validating raw HuggingFace DatasetDict')
print('=' * 60)
val_hf = ds.validate_dataset(dataset, source='hf')

# Report pass / fail per split
for split_name, result in val_hf.items():
    status = result.get('status', 'UNKNOWN')
    count  = result.get('count', 0)
    issues = result.get('issues', [])
    print(f'{split_name:>12}: [{status}]  {count:,} examples  issues={issues}')

In [ ]:
# Reload exported files via Pandas (no HF datasets required) and validate
print('=' * 60)
print('Validating exported Parquet files (Pandas reload)')
print('=' * 60)

train_df, val_df, metadata = ds.load_processed_local(processed_dir=PROCESSED_DIR)

print(f'train_df shape : {train_df.shape}')
print(f'val_df shape   : {val_df.shape}')
print(f'Columns        : {list(train_df.columns)}')

print('\nFirst 2 rows of train_df:')
print(train_df.head(2).to_string())

val_local = ds.validate_dataset((train_df, val_df), source='local')

print('\nMetadata keys:', list(metadata.keys()))
print('Splits in metadata:', metadata.get('splits', {}))

In [ ]:
# DataLoader smoke test
train_loader, val_loader = ds.get_dataloaders(
    tokenized_dataset,
    tokenizer=tokenizer,
    batch_size=config.BATCH_SIZE,
)

batch = next(iter(train_loader))

print('DataLoader smoke test:')
print(f'  Batch keys       : {list(batch.keys())}')
print(f'  input_ids shape  : {tuple(batch["input_ids"].shape)}')
print(f'  attention_mask   : {tuple(batch["attention_mask"].shape)}')
print(f'  labels (first 8) : {batch["labels"][:8].tolist()}')
print(f'  Train batches    : {len(train_loader)}')
print(f'  Val batches      : {len(val_loader)}')
assert batch['input_ids'].shape[0] == config.BATCH_SIZE, 'Batch size mismatch'
assert batch['input_ids'].shape[1] <= config.MAX_SEQ_LENGTH, 'Seq length exceeds max'
print('All DataLoader assertions passed.')

---
## Step 9 — Print validation summary

In [ ]:
print('=' * 65)
print('  DATASET PREPARATION SUMMARY')
print('=' * 65)

# Count checks
EXPECTED_TRAIN = 67349 if SUBSET_SIZE is None else SUBSET_SIZE
EXPECTED_VAL   = 872   if SUBSET_SIZE is None else SUBSET_SIZE

actual_train = len(train_df)
actual_val   = len(val_df)

count_train_ok = abs(actual_train - EXPECTED_TRAIN) <= 2
count_val_ok   = abs(actual_val   - EXPECTED_VAL)   <= 2

print(f'  Train count  : {actual_train:,}  (expected ~{EXPECTED_TRAIN:,})'
      f'  [{"PASS" if count_train_ok else "FAIL"}]')
print(f'  Val count    : {actual_val:,}   (expected ~{EXPECTED_VAL:,})'
      f'  [{"PASS" if count_val_ok else "FAIL"}]')

# Label distributions
import pandas as pd
for name, df in [('train', train_df), ('validation', val_df)]:
    dist = df['label'].value_counts().sort_index()
    pct_pos = dist.get(1, 0) / len(df) * 100
    print(f'  {name:<12} labels: neg={dist.get(0,0):,}  pos={dist.get(1,0):,}'
          f'  ({pct_pos:.1f}% positive)')

# Missing values
null_train = train_df[['example_id', 'sentence', 'label']].isnull().sum().sum()
null_val   = val_df[['example_id', 'sentence', 'label']].isnull().sum().sum()
print(f'  Missing values: train={null_train}  val={null_val}'
      f'  [{"PASS" if null_train == 0 and null_val == 0 else "FAIL"}]')

# Duplicate IDs
dup_train = train_df['example_id'].duplicated().sum()
dup_val   = val_df['example_id'].duplicated().sum()
print(f'  Duplicate IDs : train={dup_train}  val={dup_val}'
      f'  [{"PASS" if dup_train == 0 and dup_val == 0 else "FAIL"}]')

# File existence
files_ok = all(Path(str(p)).exists() for p in export_paths.values())
print(f'  Exported files: {list(export_paths.keys())}  [{"PASS" if files_ok else "FAIL"}]')

# Tokenisation
print(f'  Tokenisation  : [{"PASS" if ok else "FAIL"}]')

all_pass = all([
    count_train_ok, count_val_ok,
    null_train == 0, null_val == 0,
    dup_train == 0, dup_val == 0,
    files_ok, ok,
])
print('=' * 65)
print(f'  OVERALL: {"ALL CHECKS PASSED" if all_pass else "SOME CHECKS FAILED — review above"}')
print('=' * 65)

---
## Step 10 — Package and download processed data

In [ ]:
import shutil, os

ZIP_OUTPUT = '/content/processed_data'

# List files to be zipped
files_to_zip = list(PROCESSED_DIR.iterdir())
print('Files to include in zip:')
for f in files_to_zip:
    if f.is_file() and not f.name.startswith('.'):
        size_kb = f.stat().st_size / 1024
        print(f'  {f.name:<35} {size_kb:.1f} KB')

# Create zip — root_dir so the zip contains files at top level, not nested
zip_path = shutil.make_archive(
    ZIP_OUTPUT,          # output path without extension
    'zip',
    root_dir=str(PROCESSED_DIR),   # zip from inside processed_dir
    base_dir='.',                   # include everything in it
)
zip_size_kb = os.path.getsize(zip_path) / 1024
print(f'\nCreated: {zip_path}  ({zip_size_kb:.0f} KB)')

if IN_COLAB:
    from google.colab import files as _colab_files
    _colab_files.download(zip_path)
    print('\nDownload started.')
    print('Extract to your local data/processed/ directory.')
else:
    print(f'\nLocal run: zip saved to {zip_path}')
    print(f'Extract it to data/processed/ if needed.')

---
## Step 11 — What to do after downloading

1. Extract `processed_data.zip` on your local machine.

2. Place the three files into:
   ```
   knowledge-distillation-quantization/data/processed/
       train.parquet
       validation.parquet
       metadata.json
   ```
   (If Parquet export failed, you will have `.csv` files instead — both work.)

3. Verify locally (no Colab needed):
   ```python
   from src.dataset import load_processed_local
   train_df, val_df, meta = load_processed_local()
   print(train_df.shape, val_df.shape)   # (67349, 5) (872, 5)
   ```

4. Open `notebooks/01_teacher_training.ipynb` for the next phase.

---

### Exported file schema

| Column | Type | Description |
|--------|------|-------------|
| `example_id` | int | Official GLUE SST-2 example index |
| `sentence` | str | Raw sentence text |
| `label` | int | 0 = negative, 1 = positive |
| `split` | str | `'train'` or `'validation'` |
| `label_name` | str | `'negative'` or `'positive'` |

Token sequences are **not** stored. They are computed on-the-fly at training
time using `DataCollatorWithPadding`, which pads only to the longest sequence
in each batch, reducing memory usage.
